<a href="https://colab.research.google.com/github/norewyx0205/vlm-event-boundary/blob/main/notebooks/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ladder Event Boundary Evaluation on Colab

This notebook keeps the baseline sanity-check evaluation and runs the 6-level ladder experiment with Qwen3-VL.


In [1]:
%cd /content
!ls

/content
sample_data


In [2]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

## Clone or Update Repository

If the repository already exists in Colab, this cell pulls the latest code. If it does not exist, it clones the repo.


In [3]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = os.environ["GH_TOKEN"]
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; pulling latest changes...")
    %cd {REPO_DIR}
    !git

# check repo version
!git -C /content/vlm-event-boundary rev-parse --short HEAD
!grep -n "subprocess.Popen" /content/vlm-event-boundary/notebooks/colab_eval.ipynb


Cloning into '/content/vlm-event-boundary'...
remote: Enumerating objects: 3507, done.
remote: Counting objects: 100% (353/353), done.
remote: Compressing objects: 100% (239/239), done.
remote: Total 3507 (delta 136), reused 320 (delta 109), pack-reused 3154 (from 1)
Receiving objects: 100% (3507/3507), 95.06 MiB | 1.42 MiB/s, done.
Resolving deltas: 100% (1781/1781), done.
Updating files: 100% (2309/2309), done.
e55aa92
8758:        "    process = subprocess.Popen(\n",


In [4]:
%cd /content/vlm-event-boundary
!ls

/content/vlm-event-boundary
analysis			notebooks    scripts
baseline_boundary_videos	README.md    synthetic_boundary_videos
data				results
generate_2d_boundary_videos.py	run_eval.py


## Install Dependencies

These packages are needed for Qwen video input, video generation, and result analysis.


In [5]:
!pip install "transformers==5.9.0" accelerate "qwen-vl-utils==0.0.14" "decord==0.6.0" opencv-python imageio-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 73.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [6]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")


CUDA available: True


## Configuration

Qwen3-VL is the default model for the ladder experiment. You can change the model string here if needed.


In [7]:
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"  # Qwen3-VL-8B-Instruct commit used for reproducible runs.
EVAL_SEED = 42
ATTN_IMPLEMENTATION = "eager"
RESULT_DIR = "/content/vlm-event-boundary/results"

BASELINE_ANNOTATION = "/content/vlm-event-boundary/baseline_boundary_videos/annotations.jsonl"
SYNTHETIC_ANNOTATION = "/content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl"
LADDER_ROOT = "/content/vlm-event-boundary/data/ladder_v2"
DATASET_VERSION = "ladder_v2"

## Generate Baseline and Synthetic Reference Datasets

This regenerates the legacy simple baseline and the harder synthetic reference set. These are kept as reference points outside the 6-level ladder.

In [8]:
# Video generation is disabled for normal evaluation runs.
# Uncomment the next line when the baseline/synthetic datasets need to be regenerated.
# !python generate_2d_boundary_videos.py --dataset all

## Baseline Sanity Check

This keeps the earlier simple baseline. It should verify that Qwen3 can solve the easy before/after task.


In [9]:
from pathlib import Path

for name, annotation in [
    ("baseline", BASELINE_ANNOTATION),
    ("synthetic", SYNTHETIC_ANNOTATION),
]:
    path = Path(annotation)
    print(f"{name} annotation exists:", path.exists())
    if path.exists():
        print(f"{name} eval rows:", sum(1 for _ in open(path)))
        print(f"{name} videos:", len(list((path.parent / "videos").glob("*.mp4"))))
    else:
        print(f"{name} files not found; run the generation cell above.")

baseline annotation exists: True
baseline eval rows: 40
baseline videos: 20
synthetic annotation exists: True
synthetic eval rows: 240
synthetic videos: 120


In [10]:
!python scripts/run_eval.py \
  --annotation_path "$BASELINE_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name baseline_qwen3_sanity_check \
  --output_dir "$RESULT_DIR"

config.json: 100% 1.47k/1.47k [00:00<00:00, 2.77MB/s]
model.safetensors.index.json: 100% 67.8k/67.8k [00:00<00:00, 106MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/9.92G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/17.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/17.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   9% 1.66G/17.5G [00:03<00:41, 386MB/s, 54.2MB/s  ]
Reconstructing (incomplete total...):  14% 2.47G/17.5G [00:08<01:23, 181MB/s, 86.0MB/s  ]
Reconstructing (incomplete total...):  27% 4.75G/17.5G [00:20<01:44, 122MB/s, 86.3MB/s  ]
Reconstructing (incomplete total...):  36% 6.22G/17.5G [00:20<00:12, 903MB/s, 80.9MB/s  ]
Reconstructing (incomplete total...):  82% 14.4G/17.5G [00:47<00:13, 228MB/s,  107MB/s  ]
Reconstructing (incomplete total...):  85% 14.9G/17.5G [00:48<00:10, 251MB/s,  106M

## Synthetic Hard Reference Evaluation

This evaluates the legacy harder synthetic set so it can be compared with the simple baseline and the ladder levels.

In [11]:
!python scripts/run_eval.py \
  --annotation_path "$SYNTHETIC_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name synthetic_qwen3_reference \
  --output_dir "$RESULT_DIR"

Loading weights: 100% 750/750 [00:05<00:00, 133.51it/s]

Running synthetic_qwen3_reference from /content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl
Processing sample_001_low_boundary.mp4::prompt_original
qwen-vl-utils using torchcodec to read video.
sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing sample_001_low_boundary.mp4::prompt_swapped
sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing sample_002_low_boundary.mp4::prompt_original
sample_002_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing sample_002_low_boundary.mp4::prompt_swapped
sample_002_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing sample_003_low_boundary.mp4::prompt_original
sample_003_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing sample_003_low_boundary.mp4::prompt_swapped
sample_003_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing samp

## Generate 6-Level Ladder Dataset

This creates the 6-level `data/ladder_v2/level_*` dataset with evaluation-level mirrored annotations. Re-run this cell when generation parameters change.


In [12]:
# Video generation is disabled for normal evaluation runs.
# Uncomment this command block when the ladder dataset needs to be regenerated.
# !python scripts/generate_ladder_dataset.py \
#   --dataset_version "$DATASET_VERSION" \
#   --samples_per_level 30 \
#   --output_root "$LADDER_ROOT" \
#   --seed 42


## Check Ladder Dataset

Each level should contain 30 base samples × 4 boundary conditions × 2 mirrored prompts = 240 evaluation rows. The full ladder has 6 levels.


In [13]:
from pathlib import Path

for ann in sorted(Path(LADDER_ROOT).glob("level_*/annotations.jsonl")):
    video_count = len(list((ann.parent / "videos").glob("*.mp4")))
    row_count = sum(1 for _ in open(ann))
    print(ann.parent.name, "videos=", video_count, "eval_rows=", row_count)


level_1_simple videos= 120 eval_rows= 240
level_2_randomized videos= 120 eval_rows= 240
level_3_non_target_static_distractors videos= 120 eval_rows= 240
level_4_target_like_static_distractors videos= 120 eval_rows= 240
level_5_target_like_moving_distractors videos= 120 eval_rows= 240
level_6_hard_temporal_interference videos= 120 eval_rows= 240


## Qwen3 Ladder Smoke Test

Run a tiny subset before launching the full ladder evaluation.


In [14]:
!python scripts/run_eval.py \
  --annotation_path "$LADDER_ROOT/level_1_simple/annotations.jsonl" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name smoke_ladder_v2_level_1_simple_qwen3 \
  --output_dir "$RESULT_DIR" \
  --max_samples 4

Loading weights: 100% 750/750 [00:05<00:00, 131.79it/s]

Running smoke_ladder_v2_level_1_simple_qwen3 from /content/vlm-event-boundary/data/ladder_v2/level_1_simple/annotations.jsonl
Processing level_1_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
level_1_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_low_boundary_swapped
level_1_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_temporal_boundary_original
level_1_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_temporal_boundary_swapped
level_1_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'

Saved raw results to /content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/smoke_ladder_v2_level_1_simple_qwen3/20260803_042826/raw_results.jsonl
Saved summary to /content/vlm-event-boundary/results/Qwen_Qwen3-VL

## Run Qwen3 on All 6 Ladder Levels

This is the main ladder experiment. Results are saved under `results/<safe_model_name>/<dataset_name>/<timestamp>/`.


In [15]:
!python scripts/run_eval.py \
  --annotation_root "$LADDER_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$DATASET_VERSION"_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 132.11it/s]

Running ladder_v2_level_1_simple from /content/vlm-event-boundary/data/ladder_v2/level_1_simple/annotations.jsonl
Processing level_1_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
level_1_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_low_boundary_swapped
level_1_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_temporal_boundary_original
level_1_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_temporal_boundary_swapped
level_1_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_visual_boundary_original
level_1_sample_001_visual_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_visual_boundary_swapped
level_1_sample_001_visual_boundary.mp4 p

## Analyze Ladder Results

This aggregates all Qwen3 ladder runs and treats prompt Accuracy and Strict both-correct pair accuracy as co-primary metrics. It produces both 6-level curves, direct Accuracy-vs-Strict comparisons, paired boundary comparisons, and swap-consistency diagnostics.



In [16]:
ANALYSIS_DIR = f"analysis/{DATASET_VERSION}_ladder"

!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "ladder_v2_level_" \
  --output_dir "$ANALYSIS_DIR" \
  --plots

Analyzed 1440 rows from 6 raw result file(s).
Saved analysis to analysis/ladder_v2_ladder


## Inspect Saved Files


In [17]:
!find "$RESULT_DIR" -maxdepth 4 -type f | sort | tail -60
!find analysis -maxdepth 2 -type f | sort


/content/vlm-event-boundary/results/qwen2vl_2b/ladder_v1/.gitkeep
/content/vlm-event-boundary/results/qwen3vl/ladder_v1/.gitkeep
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/baseline_qwen3_sanity_check/20260803_041534/config.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/baseline_qwen3_sanity_check/20260803_041534/raw_results.jsonl
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/baseline_qwen3_sanity_check/20260803_041534/summary.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260803_042848/config.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260803_042848/raw_results.jsonl
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260803_042848/summary.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_2_randomized/20260803_042848/config.json
/content/vlm-event-boundary/results/Qwe

## Level 5 Feature-Ablation Pilot

This pilot compares four structurally paired Level 5 variants: full, shape-only, color-only, and size-only. It also includes a separate size-only 2x2 stress pilot crossing absolute target size with distractor count.


In [18]:
ABLATION_VERSION = "l5_feature_ablation_v1"
ABLATION_ROOT = f"/content/vlm-event-boundary/data/{ABLATION_VERSION}"
ABLATION_ANALYSIS_DIR = f"/content/vlm-event-boundary/analysis/{ABLATION_VERSION}"
SIZE_STRESS_ROOT = f"{ABLATION_ROOT}/size_stress_pilot"
SIZE_STRESS_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_stress"
SIZE_CLEAR_CONTRAST_ROOT = f"{ABLATION_ROOT}/size_clear_contrast_pilot"
SIZE_CLEAR_CONTRAST_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_clear_contrast"
DIAGNOSTIC_ROOT = "/content/vlm-event-boundary/data/diagnostics"
SIZE_CLEAR_DIAGNOSTIC_ANNOTATION = f"{DIAGNOSTIC_ROOT}/l5_size_clear_contrast_diagnostics/annotations.jsonl"
SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_clear_contrast_diagnostics"
PERTURBATION_ROOT = "/content/vlm-event-boundary/data/perturbations/l5_clear_small_many"
PERTURBATION_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_perturb_l5_clear_small_many"
ATTENTION_OUTPUT_PATH = "/content/vlm-event-boundary/analysis/attention/l5_clear_small_many_attention.json"
ATTENTION_VISUALIZATION_DIR = "/content/vlm-event-boundary/analysis/attention/l5_clear_small_many_figures"
ABLATION_VARIANTS = ["L5_full", "L5_shape_only", "L5_color_only", "L5_size_only"]


### Generate And Validate Paired Stimuli


In [19]:
# Video generation is disabled for normal evaluation runs.
# Uncomment this command block when the L5 ablation dataset needs to be regenerated.
# !python scripts/generate_l5_feature_ablation.py \
#   --dataset_version "$ABLATION_VERSION" \
#   --samples_per_variant 30 \
#   --size_stress_samples_per_cell 10 \
#   --output_root "$ABLATION_ROOT" \
#   --seed 42
#
# !python scripts/check_l5_feature_ablation.py --root "$ABLATION_ROOT"


### Run Qwen3 Evaluation

Each variant retains all four boundary conditions and original/swapped mirrored prompts.


In [20]:
!python scripts/run_eval.py \
  --annotation_root "$ABLATION_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$ABLATION_VERSION"_main_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 130.72it/s]

Running l5_feature_ablation_v1_main_L5_color_only from /content/vlm-event-boundary/data/l5_feature_ablation_v1/L5_color_only/annotations.jsonl
Processing l5_color_only_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
l5_color_only_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_color_only_sample_001_low_boundary_swapped
l5_color_only_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_color_only_sample_001_temporal_boundary_original
l5_color_only_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_color_only_sample_001_temporal_boundary_swapped
l5_color_only_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_color_only_sample_001_visual_boundary_original
l5_color_only_sample_001_visual_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Proce

### Run Size-Only 2x2 Stress Pilot

This evaluates 10 base samples in each of large/few, large/many, small/few, and small/many, for 320 prompt evaluations in total.


In [21]:
!python scripts/run_eval.py \
  --annotation_root "$SIZE_STRESS_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$ABLATION_VERSION"_size_stress_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 133.05it/s]

Running l5_feature_ablation_v1_size_stress_L5_size_only_large_few from /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_stress_pilot/L5_size_only_large_few/annotations.jsonl
Processing l5_size_only_large_few_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
l5_size_only_large_few_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_size_only_large_few_sample_001_low_boundary_swapped
l5_size_only_large_few_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_size_only_large_few_sample_001_temporal_boundary_original
l5_size_only_large_few_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_size_only_large_few_sample_001_temporal_boundary_swapped
l5_size_only_large_few_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_size_only_large_few_sample_

### Analyze Feature, Boundary, And Position Effects

The analysis uses the latest run for each variant and writes prompt accuracy, strict mirrored-pair accuracy, the accuracy-strict gap `d`, position-sensitive pair rates, paired comparisons, swap consistency, and report-ready plots for the main research questions.


In [22]:
!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "$ABLATION_VERSION"_main_ \
  --latest_per_dataset \
  --output_dir "$ABLATION_ANALYSIS_DIR" \
  --plots

!find "$ABLATION_ANALYSIS_DIR" -maxdepth 1 -type f | sort


Analyzed 960 rows from 4 raw result file(s).
Saved analysis to /content/vlm-event-boundary/analysis/l5_feature_ablation_v1
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_difficulty_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_difficulty_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_difficulty.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_prompt_variant

### Analyze Size And Crowding Effects

This produces cell-level accuracy, strict mirrored-pair accuracy, boundary-condition plots, and the large-vs-small, few-vs-many, and interaction estimates.


In [23]:
!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "$ABLATION_VERSION"_size_stress_ \
  --latest_per_dataset \
  --output_dir "$SIZE_STRESS_ANALYSIS_DIR" \
  --plots

!find "$SIZE_STRESS_ANALYSIS_DIR" -maxdepth 1 -type f | sort


Analyzed 320 rows from 4 raw result file(s).
Saved analysis to /content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_difficulty_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_difficulty_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_difficulty.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_prompt_variant.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_size_scene_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_size_scene_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_size_scene_correct_option.csv
/content/v

### Generate Size-Only Clear-Contrast Pilot

This is the final Part 1 size-only contrast check. It repeats the 2x2 size/crowding design with clearer target-distractor size margins so we can test whether the previous size-only pattern survives when the smallest/largest distinction is visually obvious.


In [24]:
!python scripts/generate_l5_feature_ablation.py \
  --dataset_version "$ABLATION_VERSION" \
  --size_clear_contrast_only \
  --size_clear_contrast_samples_per_cell 10 \
  --output_root "$ABLATION_ROOT" \
  --seed 42

!python scripts/check_l5_feature_ablation.py --root "$ABLATION_ROOT"


L5_size_only_clear_large_few: wrote 80 eval rows to /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_clear_contrast_pilot/L5_size_only_clear_large_few/annotations.jsonl
L5_size_only_clear_large_many: wrote 80 eval rows to /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_clear_contrast_pilot/L5_size_only_clear_large_many/annotations.jsonl
L5_size_only_clear_small_few: wrote 80 eval rows to /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_clear_contrast_pilot/L5_size_only_clear_small_few/annotations.jsonl
L5_size_only_clear_small_many: wrote 80 eval rows to /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_clear_contrast_pilot/L5_size_only_clear_small_many/annotations.jsonl
L5_full: eval_rows=240 videos=120 disk_videos=120
L5_shape_only: eval_rows=240 videos=120 disk_videos=120
L5_color_only: eval_rows=240 videos=120 disk_videos=120
L5_size_only: eval_rows=240 videos=120 disk_videos=120
paired_stimuli=120
L5_size_only_large_few: eval_rows=80

### Run Size-Only Clear-Contrast Pilot

This evaluates four clear-contrast scenes: clear_large_few, clear_large_many, clear_small_few, and clear_small_many. Each scene has 10 base samples, four boundary conditions, and original/swapped mirrored prompts.


In [25]:
!python scripts/run_eval.py \
  --annotation_root "$SIZE_CLEAR_CONTRAST_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$ABLATION_VERSION"_size_clear_contrast_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 131.80it/s]

Running l5_feature_ablation_v1_size_clear_contrast_L5_size_only_clear_large_few from /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_clear_contrast_pilot/L5_size_only_clear_large_few/annotations.jsonl
Processing l5_size_only_clear_large_few_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
l5_size_only_clear_large_few_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_size_only_clear_large_few_sample_001_low_boundary_swapped
l5_size_only_clear_large_few_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_size_only_clear_large_few_sample_001_temporal_boundary_original
l5_size_only_clear_large_few_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_size_only_clear_large_few_sample_001_temporal_boundary_swapped
l5_size_only_clear_large_few_sample_001_temporal_boundary.mp4 pred= B co

### Analyze Clear-Contrast Size Effects

This produces the same accuracy, strict both-correct, boundary, size/crowding, and interaction summaries as the original size stress pilot, but for the clearer size contrast stimuli.


In [26]:
!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "$ABLATION_VERSION"_size_clear_contrast_ \
  --latest_per_dataset \
  --output_dir "$SIZE_CLEAR_CONTRAST_ANALYSIS_DIR" \
  --plots

!find "$SIZE_CLEAR_CONTRAST_ANALYSIS_DIR" -maxdepth 1 -type f | sort


Analyzed 320 rows from 4 raw result file(s).
Saved analysis to /content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_difficulty_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_difficulty_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_difficulty.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_prompt_variant.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_size_scene_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_clear_contrast/accuracy_by_size_scene_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_

## Mechanism Probes: Diagnostics, Perturbation, And Attention

These cells are optional and should be run after the clear-contrast evaluation. They implement the two-layer method: first behavioral diagnostics and causal perturbations, then small-sample attention/ROI probing. They are disabled by default so a normal Run All does not accidentally add a long diagnostic run.


### Create Diagnostic Prompts

Diagnostic prompts reuse the same videos but ask simpler questions about object identity, motion binding, and first-mover identity. This helps separate `recognise objects` from `track objects` and final before/after reasoning.


In [27]:
import subprocess

RUN_DIAGNOSTIC_EVAL = True

if RUN_DIAGNOSTIC_EVAL:
    subprocess.run([
        "python", "scripts/make_diagnostic_annotations.py",
        "--annotation_root", SIZE_CLEAR_CONTRAST_ROOT,
        "--output_path", SIZE_CLEAR_DIAGNOSTIC_ANNOTATION,
    ], check=True)
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", SIZE_CLEAR_DIAGNOSTIC_ANNOTATION,
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "l5_size_clear_contrast_diagnostics",
        "--output_dir", RESULT_DIR,
    ], check=True)
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", "l5_size_clear_contrast_diagnostics",
        "--latest_per_dataset",
        "--output_dir", SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR,
        "--plots",
    ], check=True)
    subprocess.run([
        "find", SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR,
        "-maxdepth", "1", "-type", "f",
    ], check=True)
else:
    print("Set RUN_DIAGNOSTIC_EVAL = True to create and run diagnostic prompts.")


Set RUN_DIAGNOSTIC_EVAL = True to create and run diagnostic prompts.


### Causal ROI Perturbation Probe

This creates masked-video variants for a small subset of `clear_small_many`, then evaluates whether masking targets, distractors, or the boundary region changes model behavior. First enable `RUN_PERTURBATION_BUILD` and inspect the QA sheet; only then enable `RUN_PERTURBATION_EVAL`. This is stronger evidence than attention alone because it tests what the answer depends on.


In [28]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

RUN_PERTURBATION_BUILD = True
RUN_PERTURBATION_EVAL = True
PERTURBATION_SOURCE = f"{SIZE_CLEAR_CONTRAST_ROOT}/L5_size_only_clear_small_many/annotations.jsonl"
PERTURBATION_MAX_BASE_SAMPLES = 10
PERTURBATION_TYPES = "original,mask_target_1,mask_target_2,mask_distractors,remove_visual_marker"
ROI_MASK_PADDING = 6
ROI_MASK_SCOPE = "all_frames"  # use motion_window to isolate motion evidence

if RUN_PERTURBATION_BUILD:
    subprocess.run([
        "python", "scripts/make_roi_perturbation_dataset.py",
        "--annotation_path", PERTURBATION_SOURCE,
        "--output_root", PERTURBATION_ROOT,
        "--max_base_samples", str(PERTURBATION_MAX_BASE_SAMPLES),
        "--perturbations", PERTURBATION_TYPES,
        "--mask_padding", str(ROI_MASK_PADDING),
        "--mask_mode", "dynamic",
        "--mask_scope", ROI_MASK_SCOPE,
    ], check=True)
    PERTURBATION_PREVIEW_PATH = f"{PERTURBATION_ANALYSIS_DIR}/roi_qa_preview.png"
    subprocess.run([
        "python", "scripts/visualize_roi_perturbations.py",
        "--annotation_path", f"{PERTURBATION_ROOT}/annotations.jsonl",
        "--condition", "visual_boundary",
        "--output_path", PERTURBATION_PREVIEW_PATH,
    ], check=True)
    display(Image(filename=PERTURBATION_PREVIEW_PATH))

if RUN_PERTURBATION_EVAL:
    perturbation_annotation = Path(PERTURBATION_ROOT) / "annotations.jsonl"
    if not perturbation_annotation.exists():
        raise FileNotFoundError("Build and inspect the ROI perturbation dataset first.")
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", f"{PERTURBATION_ROOT}/annotations.jsonl",
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "l5_clear_small_many_perturbation",
        "--output_dir", RESULT_DIR,
    ], check=True)
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", "l5_clear_small_many_perturbation",
        "--latest_per_dataset",
        "--output_dir", PERTURBATION_ANALYSIS_DIR,
        "--plots",
    ], check=True)
if not RUN_PERTURBATION_BUILD and not RUN_PERTURBATION_EVAL:
    print("Set RUN_PERTURBATION_BUILD = True to generate/preview masks, then RUN_PERTURBATION_EVAL = True to evaluate them.")


Set RUN_PERTURBATION_BUILD = True to generate/preview masks, then RUN_PERTURBATION_EVAL = True to evaluate them.


### Attention ROI Probe

This uses a cached single-token decoder probe to avoid materializing full prompt-by-prompt attention matrices. It maps answer-token attention back to the spatially merged video grid and writes frame overlays, temporal profiles, and layer-wise ROI-enrichment heatmaps. Use these visualizations as qualitative mechanism probes alongside the behavioral and perturbation results, not as standalone explanations.


In [29]:
import json
import subprocess
from pathlib import Path
from IPython.display import Image, display

RUN_ATTENTION_ROI_PROBE = True
ATTENTION_SOURCE = f"{SIZE_CLEAR_CONTRAST_ROOT}/L5_size_only_clear_small_many/annotations.jsonl"
ATTENTION_MAX_SAMPLES = 8
ATTENTION_CONDITIONS = "temporal_boundary,visual_boundary"
ATTENTION_PROMPT_VARIANTS = "original,swapped"

if RUN_ATTENTION_ROI_PROBE:
    attention_command = [
        "python", "scripts/probe_attention_roi.py",
        "--annotation_path", ATTENTION_SOURCE,
        "--output_path", ATTENTION_OUTPUT_PATH,
        "--visualization_dir", ATTENTION_VISUALIZATION_DIR,
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", "eager",
        "--max_samples", str(ATTENTION_MAX_SAMPLES),
        "--conditions", ATTENTION_CONDITIONS,
        "--prompt_variants", ATTENTION_PROMPT_VARIANTS,
        "--roi_padding", "8",
        "--visualization_layer", "-1",
        "--head_reduction", "mean",
        "--empty_cache_each_sample",
    ]
    process = subprocess.Popen(
        attention_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Attention probe failed with exit code {return_code}; see the full log above."
        )
    rows = json.loads(Path(ATTENTION_OUTPUT_PATH).read_text())
    for row in rows[:3]:
        print(
            row.get("eval_id"),
            "prediction=", row.get("prediction"),
            "video_attention=", row.get("selected_layer_visual_attention_fraction"),
            "spatial_roi=", row.get("spatial_roi_attention"),
        )
    for figure_path in sorted(Path(ATTENTION_VISUALIZATION_DIR).glob("*.png"))[:6]:
        display(Image(filename=str(figure_path)))
else:
    print("Set RUN_ATTENTION_ROI_PROBE = True to run a small attention/ROI probe.")


Set RUN_ATTENTION_ROI_PROBE = True to run a small attention/ROI probe.


## Download Timestamped Experiment Archive

Run this after an experiment. It packages all saved evaluation results, analyses, and ablation annotations into a timestamped ZIP and downloads it to your local computer. Videos are excluded to keep the archive manageable.


In [30]:
from datetime import datetime
import json
from pathlib import Path
import zipfile
from google.colab import files

project_root = Path("/content/vlm-event-boundary")
archive_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = Path("/content") / f"vlm_event_boundary_results_{archive_timestamp}.zip"
manifest = {
    "created_at": archive_timestamp,
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION or None,
    "eval_seed": EVAL_SEED,
    "deterministic": True,
    "attention_implementation": ATTN_IMPLEMENTATION,
    "ladder_version": DATASET_VERSION,
    "ablation_version": globals().get("ABLATION_VERSION"),
}

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("archive_manifest.json", json.dumps(manifest, indent=2))
    for folder in [Path(RESULT_DIR), project_root / "analysis"]:
        if folder.exists():
            for path in folder.rglob("*"):
                if path.is_file():
                    archive.write(path, path.relative_to(project_root))
    ablation_root = Path(globals().get("ABLATION_ROOT", ""))
    if ablation_root.exists():
        for pattern in ["annotations.jsonl", "config.json", "README.md"]:
            for path in ablation_root.rglob(pattern):
                archive.write(path, path.relative_to(project_root))
    for extra_data_root in [project_root / "data" / "diagnostics", project_root / "data" / "perturbations"]:
        if extra_data_root.exists():
            for pattern in ["annotations.jsonl", "manifest.json", "perturbation_stats.jsonl"]:
                for path in extra_data_root.rglob(pattern):
                    archive.write(path, path.relative_to(project_root))

print(f"Created {archive_path} ({archive_path.stat().st_size / 1024 / 1024:.1f} MB)")
files.download(str(archive_path))


Created /content/vlm_event_boundary_results_20260803_061308.zip (1.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>